### Ноутбук 3. Моделирование и оптимизация (версия 2)

**Цель:** Продолжить поиск наиболее точной модели с использованием валидационной выборки и дополнительных методов.

**Отличия от первой версии:**

1. **Валидационная выборка** — разделение train / validation / test (60% / 20% / 20%)
2. **Логарифмирование целевой переменной** — для нормализации распределения
3. **Стандартизация** — проверка влияния на качество моделей
4. **Удаление выбросов из target** — для устойчивости
5. **Подбор гиперпараметров** — GridSearchCV и Optuna
6. **Ансамблевые методы** — Stacking Regressor
7. **Снижение размерности** — PCA

**Задача:** Превзойти результат R² = 0.6222 или найти модель с лучшим MAE.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Настройка рабочей директории
def get_project_root():
    current = os.getcwd()
    root = current
    while True:
        if os.path.exists(os.path.join(root, 'data_processed')):
            break
        new_root = os.path.dirname(root)
        if new_root == root:
            root = current
            break
        root = new_root
    os.chdir(root)
    print(f"Корень проекта: {root}")
    return root

PROJECT_ROOT = get_project_root()

# Настройка визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Загрузка данных
df = pd.read_csv('data_processed/data_features.csv')

print("\n=== ЗАГРУЗКА ДАННЫХ ===")
print(f"Размер: {df.shape[0]:,} строк, {df.shape[1]} колонок")
print(f"Колонки: {df.columns.tolist()}")

Корень проекта: c:\Users\washe\Documents\SF_Training_DS\Диплом

=== ЗАГРУЗКА ДАННЫХ ===
Размер: 356,495 строк, 23 колонок
Колонки: ['status', 'propertyType', 'baths', 'fireplace', 'city', 'sqft', 'zipcode', 'beds', 'state', 'stories', 'target', 'has_fireplace_info', 'year_built', 'heating', 'parking', 'lot_size', 'schools_count', 'avg_school_rating', 'nearest_school_dist', 'street_group', 'house_age', 'is_land', 'rooms_per_1000sqft']


### 2. Подготовка данных

**Шаги:**
1. Кодирование категориальных признаков
2. Разделение на train / validation / test (60% / 20% / 20%)
3. Создание версии со стандартизацией и без для сравнения

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=== ПОДГОТОВКА ДАННЫХ ===\n")

# Целевая переменная
target_col = 'target'
y = df[target_col]

# Признаки (все, кроме target)
feature_cols = [col for col in df.columns if col != target_col]
X = df[feature_cols]

print(f"Целевая переменная: {target_col}")
print(f"Количество признаков: {X.shape[1]}")

# Кодирование категориальных признаков
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"\nКатегориальные признаки ({len(categorical_cols)}): {categorical_cols}")

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"  {col}: {len(le.classes_)} категорий")

# Разделение на train (60%), validation (20%), test (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"\n=== РАЗМЕРЫ ВЫБОРОК ===")
print(f"Train: {X_train.shape[0]:,} объектов")
print(f"Validation: {X_val.shape[0]:,} объектов")
print(f"Test: {X_test.shape[0]:,} объектов")

=== ПОДГОТОВКА ДАННЫХ ===

Целевая переменная: target
Количество признаков: 22

Категориальные признаки (9): ['status', 'propertyType', 'fireplace', 'city', 'zipcode', 'state', 'heating', 'parking', 'street_group']
  status: 7 категорий
  propertyType: 504 категорий
  fireplace: 1629 категорий
  city: 1951 категорий
  zipcode: 4505 категорий
  state: 38 категорий
  heating: 1911 категорий
  parking: 3232 категорий
  street_group: 192 категорий

=== РАЗМЕРЫ ВЫБОРОК ===
Train: 213,897 объектов
Validation: 71,299 объектов
Test: 71,299 объектов


In [3]:
print("=== СОЗДАНИЕ ВЕРСИЙ ДАННЫХ ===\n")

# Копии без стандартизации
X_train_raw = X_train.copy()
X_val_raw = X_val.copy()
X_test_raw = X_test.copy()

# Стандартизация числовых признаков
# Определяем числовые колонки (все, кроме закодированных категорий)
numeric_cols = ['baths', 'sqft', 'beds', 'stories', 'year_built', 'lot_size',
                'schools_count', 'avg_school_rating', 'nearest_school_dist',
                'house_age', 'rooms_per_1000sqft', 'has_fireplace_info', 'is_land']

# Берём только те, что есть в данных
numeric_cols_present = [col for col in numeric_cols if col in X_train.columns]

print(f"Числовые признаки для стандартизации: {numeric_cols_present}")

# Создаём scaler и применяем
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols_present] = scaler.fit_transform(X_train[numeric_cols_present])
X_val_scaled[numeric_cols_present] = scaler.transform(X_val[numeric_cols_present])
X_test_scaled[numeric_cols_present] = scaler.transform(X_test[numeric_cols_present])

print("\n✅ Созданы версии:")
print("   - X_train_raw, X_val_raw, X_test_raw (без стандартизации)")
print("   - X_train_scaled, X_val_scaled, X_test_scaled (со стандартизацией)")
print(f"\n   Scaler сохранён для возможного использования в веб-сервисе")

=== СОЗДАНИЕ ВЕРСИЙ ДАННЫХ ===

Числовые признаки для стандартизации: ['baths', 'sqft', 'beds', 'stories', 'year_built', 'lot_size', 'schools_count', 'avg_school_rating', 'nearest_school_dist', 'house_age', 'rooms_per_1000sqft', 'has_fireplace_info', 'is_land']

✅ Созданы версии:
   - X_train_raw, X_val_raw, X_test_raw (без стандартизации)
   - X_train_scaled, X_val_scaled, X_test_scaled (со стандартизацией)

   Scaler сохранён для возможного использования в веб-сервисе


### 4. Базовые модели (без стандартизации)

**Обучаем и сравниваем:**
- Linear Regression
- SGDRegressor
- Decision Tree Regressor

**Оценка:** MAE, RMSE, R² на валидационной выборке

In [4]:
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
import time

print("=== БАЗОВЫЕ МОДЕЛИ (БЕЗ СТАНДАРТИЗАЦИИ) ===\n")

models_raw = {
    'Linear Regression': LinearRegression(),
    'SGDRegressor': SGDRegressor(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42)
}

results_raw = []

for name, model in models_raw.items():
    start = time.time()
    model.fit(X_train_raw, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_val_raw)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)

    results_raw.append({
        'Model': name,
        'MAE': f"${mae:,.0f}",
        'RMSE': f"${rmse:,.0f}",
        'R²': f"{r2:.4f}",
        'Time (s)': f"{train_time:.2f}"
    })

    print(f"\n{name}:")
    print(f"  MAE: ${mae:,.0f}")
    print(f"  RMSE: ${rmse:,.0f}")
    print(f"  R²: {r2:.4f}")
    print(f"  Время: {train_time:.2f} сек")

print("\n" + "="*50)
print("СВОДНАЯ ТАБЛИЦА (БЕЗ СТАНДАРТИЗАЦИИ)")
print("="*50)
results_raw_df = pd.DataFrame(results_raw)
print(results_raw_df.to_string(index=False))

=== БАЗОВЫЕ МОДЕЛИ (БЕЗ СТАНДАРТИЗАЦИИ) ===


Linear Regression:
  MAE: $424,241
  RMSE: $816,230
  R²: 0.1243
  Время: 0.12 сек

SGDRegressor:
  MAE: $2,540,683,574,501,859,917,824
  RMSE: $3,238,539,788,167,052,525,568
  R²: -13785323773952308775805331177472.0000
  Время: 21.45 сек

Decision Tree:
  MAE: $298,442
  RMSE: $770,896
  R²: 0.2189
  Время: 2.28 сек

СВОДНАЯ ТАБЛИЦА (БЕЗ СТАНДАРТИЗАЦИИ)
            Model                            MAE                           RMSE                                     R² Time (s)
Linear Regression                       $424,241                       $816,230                                 0.1243     0.12
     SGDRegressor $2,540,683,574,501,859,917,824 $3,238,539,788,167,052,525,568 -13785323773952308775805331177472.0000    21.45
    Decision Tree                       $298,442                       $770,896                                 0.2189     2.28


Вывод:

Без стандартизации SGDRegressor не работает. 

Линейная регрессия и дерево решений показывают низкое качество

### 5. Базовые модели (со стандартизацией)

Обучаем те же модели на стандартизированных данных.

In [5]:
print("=== БАЗОВЫЕ МОДЕЛИ (СО СТАНДАРТИЗАЦИЕЙ) ===\n")

models_scaled = {
    'Linear Regression': LinearRegression(),
    'SGDRegressor': SGDRegressor(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42)
}

results_scaled = []

for name, model in models_scaled.items():
    start = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_val_scaled)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)

    results_scaled.append({
        'Model': name,
        'MAE': f"${mae:,.0f}",
        'RMSE': f"${rmse:,.0f}",
        'R²': f"{r2:.4f}",
        'Time (s)': f"{train_time:.2f}"
    })

    print(f"\n{name}:")
    print(f"  MAE: ${mae:,.0f}")
    print(f"  RMSE: ${rmse:,.0f}")
    print(f"  R²: {r2:.4f}")
    print(f"  Время: {train_time:.2f} сек")

print("\n" + "="*50)
print("СВОДНАЯ ТАБЛИЦА (СО СТАНДАРТИЗАЦИЕЙ)")
print("="*50)
results_scaled_df = pd.DataFrame(results_scaled)
print(results_scaled_df.to_string(index=False))

=== БАЗОВЫЕ МОДЕЛИ (СО СТАНДАРТИЗАЦИЕЙ) ===


Linear Regression:
  MAE: $424,241
  RMSE: $816,230
  R²: 0.1243
  Время: 0.11 сек

SGDRegressor:
  MAE: $779,447,017,323,252
  RMSE: $898,601,272,305,858
  R²: -1061335167584101120.0000
  Время: 33.29 сек

Decision Tree:
  MAE: $298,535
  RMSE: $771,069
  R²: 0.2185
  Время: 2.26 сек

СВОДНАЯ ТАБЛИЦА (СО СТАНДАРТИЗАЦИЕЙ)
            Model                  MAE                 RMSE                        R² Time (s)
Linear Regression             $424,241             $816,230                    0.1243     0.11
     SGDRegressor $779,447,017,323,252 $898,601,272,305,858 -1061335167584101120.0000    33.29
    Decision Tree             $298,535             $771,069                    0.2185     2.26


РЕзультаты показали, что StandartScaller не помогает при сильном разбросе данных.

Меняем метод стандартизации.

In [6]:
from sklearn.preprocessing import RobustScaler

print("=== СОЗДАНИЕ ВЕРСИИ С ROBUSTSCALER ===\n")

# Создаём scaler и применяем
robust_scaler = RobustScaler()
X_train_robust = X_train.copy()
X_val_robust = X_val.copy()
X_test_robust = X_test.copy()

X_train_robust[numeric_cols_present] = robust_scaler.fit_transform(X_train[numeric_cols_present])
X_val_robust[numeric_cols_present] = robust_scaler.transform(X_val[numeric_cols_present])
X_test_robust[numeric_cols_present] = robust_scaler.transform(X_test[numeric_cols_present])

print("✅ Создана версия с RobustScaler (устойчив к выбросам)")

# Обучаем SGDRegressor на RobustScaler
print("\n=== SGDREGRESSOR С ROBUSTSCALER ===")

sgd_robust = SGDRegressor(max_iter=1000, random_state=42)
sgd_robust.fit(X_train_robust, y_train)
y_pred_robust = sgd_robust.predict(X_val_robust)

mae_robust = mean_absolute_error(y_val, y_pred_robust)
rmse_robust = np.sqrt(mean_squared_error(y_val, y_pred_robust))
r2_robust = r2_score(y_val, y_pred_robust)

print(f"MAE: ${mae_robust:,.0f}")
print(f"RMSE: ${rmse_robust:,.0f}")
print(f"R²: {r2_robust:.4f}")

if r2_robust > 0:
    print(f"\n✅ RobustScaler помог! R² = {r2_robust:.4f}")
else:
    print("\n❌ RobustScaler не помог. SGDRegressor не подходит для этих данных.")

=== СОЗДАНИЕ ВЕРСИИ С ROBUSTSCALER ===

✅ Создана версия с RobustScaler (устойчив к выбросам)

=== SGDREGRESSOR С ROBUSTSCALER ===
MAE: $7,821,740,689,462,136,832
RMSE: $292,374,907,121,915,461,632
R²: -112356626302527560096151502848.0000

❌ RobustScaler не помог. SGDRegressor не подходит для этих данных.


SGDRegressor явно не подходит для этих данных. Даже с RobustScaler результат отвратительный, исключаем его из дальнейшего исследования.

### 6. Улучшенные модели

Переходим к моделям, которые лучше работают с нелинейными данными:
- Random Forest
- XGBoost
- LightGBM

In [7]:
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except ImportError:
    xgb_available = False
    print("XGBoost не установлен")

try:
    from lightgbm import LGBMRegressor
    lgbm_available = True
except ImportError:
    lgbm_available = False
    print("LightGBM не установлен")

print("=== УЛУЧШЕННЫЕ МОДЕЛИ ===\n")

# Словарь моделей
improved_models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

if xgb_available:
    improved_models['XGBoost'] = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0)

if lgbm_available:
    improved_models['LightGBM'] = LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)

results_improved = []

for name, model in improved_models.items():
    print(f"\nОбучение: {name}")
    start = time.time()
    model.fit(X_train_raw, y_train)  # используем raw (без стандартизации)
    train_time = time.time() - start

    y_pred = model.predict(X_val_raw)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)

    results_improved.append({
        'Model': name,
        'MAE': f"${mae:,.0f}",
        'RMSE': f"${rmse:,.0f}",
        'R²': f"{r2:.4f}",
        'Time (s)': f"{train_time:.1f}"
    })

    print(f"  MAE: ${mae:,.0f}")
    print(f"  R²: {r2:.4f}")

print("\n" + "="*50)
print("СВОДНАЯ ТАБЛИЦА (УЛУЧШЕННЫЕ МОДЕЛИ)")
print("="*50)
results_improved_df = pd.DataFrame(results_improved)
print(results_improved_df.to_string(index=False))

=== УЛУЧШЕННЫЕ МОДЕЛИ ===


Обучение: Random Forest
  MAE: $237,221
  R²: 0.5582

Обучение: XGBoost
  MAE: $277,789
  R²: 0.5118

Обучение: LightGBM
  MAE: $292,078
  R²: 0.4753

СВОДНАЯ ТАБЛИЦА (УЛУЧШЕННЫЕ МОДЕЛИ)
        Model      MAE     RMSE     R² Time (s)
Random Forest $237,221 $579,797 0.5582     27.2
      XGBoost $277,789 $609,422 0.5118      0.7
     LightGBM $292,078 $631,826 0.4753      0.5


### 7. Сравнение с явной стратификацией (результат 3-го ноутбука)

| Подход | R² |
|--------|-----|
| Единая модель Random Forest | 0.5582 |
| **Явная стратификация по цене** | **0.6222** |

**Вывод:** Стратификация по цене даёт улучшение на 0.064 (6.4%).

**Решение:** Использовать стратифицированную модель из 3-го ноутбука.

In [8]:
from sklearn.ensemble import StackingRegressor

print("=== СТЕКИНГ (STACKING REGRESSOR) ===\n")

# Базовые модели
base_models = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ('xgb', XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0)),
    ('lgb', LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1))
]

# Мета-модель
meta_model = LinearRegression()

# Stacking
stacking = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5  # 5-fold cross-validation
)

print("Обучаем Stacking Regressor (RF + XGB + LGB → LinearRegression)...")
start = time.time()
stacking.fit(X_train_raw, y_train)
train_time = time.time() - start

# Оценка
y_pred_stacking = stacking.predict(X_val_raw)

mae_stacking = mean_absolute_error(y_val, y_pred_stacking)
rmse_stacking = np.sqrt(mean_squared_error(y_val, y_pred_stacking))
r2_stacking = r2_score(y_val, y_pred_stacking)

print(f"\nРезультаты Stacking:")
print(f"  MAE: ${mae_stacking:,.0f}")
print(f"  RMSE: ${rmse_stacking:,.0f}")
print(f"  R²: {r2_stacking:.4f}")
print(f"  Время: {train_time:.1f} сек")

# Сравнение с Random Forest
print("\n" + "="*50)
print("СРАВНЕНИЕ С RANDOM FOREST:")
print("="*50)
print(f"Random Forest:  R² = 0.5582")
print(f"Stacking:       R² = {r2_stacking:.4f}")
print(f"Улучшение:      +{r2_stacking - 0.5582:.4f}")

=== СТЕКИНГ (STACKING REGRESSOR) ===

Обучаем Stacking Regressor (RF + XGB + LGB → LinearRegression)...

Результаты Stacking:
  MAE: $237,669
  RMSE: $568,443
  R²: 0.5753
  Время: 137.8 сек

СРАВНЕНИЕ С RANDOM FOREST:
Random Forest:  R² = 0.5582
Stacking:       R² = 0.5753
Улучшение:      +0.0171


In [14]:
print("=== ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ ===\n")

from sklearn.model_selection import train_test_split

# Копируем данные
X_final = X_train_raw.copy()
y_final = y_train.copy()

# 1. Удаляем выбросы из target (> 99.5 перцентиля)
upper_bound = y_final.quantile(0.995)
before = len(X_final)
mask = y_final <= upper_bound
X_final = X_final[mask]
y_final = y_final[mask]
print(f"1. Удалено выбросов из target > ${upper_bound:,.0f}: {before - len(X_final)} объектов")

# 2. Логарифмируем target
y_final_log = np.log1p(y_final)
print(f"2. Логарифмирование target выполнено")

# 3. Разделение на train и val
X_final_train, X_final_val, y_final_train, y_final_val = train_test_split(
    X_final, y_final_log, test_size=0.2, random_state=42
)

print(f"\nИтоговый размер train: {X_final_train.shape[0]:,}")
print(f"Итоговый размер val: {X_final_val.shape[0]:,}")

=== ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ ===

1. Удалено выбросов из target > $6,500,000: 1014 объектов
2. Логарифмирование target выполнено

Итоговый размер train: 170,306
Итоговый размер val: 42,577


In [15]:
import optuna
from optuna.samplers import TPESampler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import numpy as np

print("=== OPTUNA ДЛЯ RANDOM FOREST ===\n")

def objective_rf(trial):
    # Гиперпараметры
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 10, 50, step=5),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7])
    }

    rf = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    rf.fit(X_final_train, y_final_train)
    y_pred = rf.predict(X_final_val)

    return r2_score(y_final_val, y_pred)

print("Поиск оптимальных параметров (20 trials)...")
study_rf = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=20, show_progress_bar=True)

print(f"\nЛучшие параметры: {study_rf.best_params}")
print(f"Лучший R² на валидации: {study_rf.best_value:.4f}")

# Обучаем финальную модель
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=42, n_jobs=-1)
best_rf.fit(X_final_train, y_final_train)

# Оценка на валидации (в реальных ценах)
y_pred_log = best_rf.predict(X_final_val)
y_pred = np.expm1(y_pred_log)
y_val_original = np.expm1(y_final_val)

mae_rf_optuna = mean_absolute_error(y_val_original, y_pred)
r2_rf_optuna = r2_score(y_val_original, y_pred)

print(f"\nРезультат Random Forest + Optuna + log + удаление выбросов:")
print(f"  MAE: ${mae_rf_optuna:,.0f}")
print(f"  R²: {r2_rf_optuna:.4f}")

[I 2026-05-28 11:19:30,467] A new study created in memory with name: no-name-2b09357a-34a3-4862-b7c1-814206f74b92


=== OPTUNA ДЛЯ RANDOM FOREST ===

Поиск оптимальных параметров (20 trials)...


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-05-28 11:19:55,231] Trial 0 finished with value: 0.7090874890840239 and parameters: {'n_estimators': 250, 'max_depth': 50, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 0.7}. Best is trial 0 with value: 0.7090874890840239.
[I 2026-05-28 11:20:07,079] Trial 1 finished with value: 0.6369279734721058 and parameters: {'n_estimators': 350, 'max_depth': 40, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7090874890840239.
[I 2026-05-28 11:20:14,415] Trial 2 finished with value: 0.675693344186765 and parameters: {'n_estimators': 200, 'max_depth': 30, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7090874890840239.
[I 2026-05-28 11:20:36,093] Trial 3 finished with value: 0.7059369497450173 and parameters: {'n_estimators': 300, 'max_depth': 45, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 0.5}. Best is trial 0 with value: 0.7090874890840239.


In [16]:
print("=== ФИНАЛЬНАЯ МОДЕЛЬ С ЛУЧШИМИ ПАРАМЕТРАМИ OPTUNA ===\n")

# Лучшие параметры из Optuna
best_params = {'n_estimators': 500, 'max_depth': 35, 'min_samples_split': 10,
               'min_samples_leaf': 1, 'max_features': 0.7}

# Обучаем модель на ВСЁМ train (не только на подвыборке)
final_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
final_model.fit(X_final_train, y_final_train)

# Оцениваем на ТЕСТОВОЙ выборке (которую мы не трогали до этого)
y_pred_log = final_model.predict(X_test_raw)
y_pred = np.expm1(y_pred_log)

mae_final = mean_absolute_error(y_test, y_pred)
r2_final = r2_score(y_test, y_pred)

print(f"Результат на ТЕСТОВОЙ выборке:")
print(f"  MAE: ${mae_final:,.0f}")
print(f"  R²: {r2_final:.4f}")

print("\n" + "="*50)
print("ИТОГОВОЕ СРАВНЕНИЕ:")
print("="*50)
print(f"Стратификация (3-й ноутбук):        R² = 0.6222")
print(f"Optuna + логарифмирование:           R² = {r2_final:.4f}")

if r2_final > 0.6222:
    print("\n🏆 OPTUNA ПРЕВЗОШЁЛ СТРАТИФИКАЦИЮ!")
else:
    print("\n⚠️ Стратификация остаётся лучшей")

=== ФИНАЛЬНАЯ МОДЕЛЬ С ЛУЧШИМИ ПАРАМЕТРАМИ OPTUNA ===

Результат на ТЕСТОВОЙ выборке:
  MAE: $224,742
  R²: 0.4511

ИТОГОВОЕ СРАВНЕНИЕ:
Стратификация (3-й ноутбук):        R² = 0.6222
Optuna + логарифмирование:           R² = 0.4511

⚠️ Стратификация остаётся лучшей


## Окончательные выводы по моделированию

### 1. Сравнение подходов

| Подход | R² | MAE | Преимущества | Недостатки |
|--------|-----|-----|--------------|-------------|
| Явная стратификация по цене | 0.6222 | выше | Лучший R², стабильность | MAE выше |
| Единая модель Random Forest | 0.5582 | $237,221 | Простота | Низкий R² |
| Stacking (RF+XGB+LGB) | 0.5753 | $237,669 | Хороший R² | Сложность |
| Optuna + логарифмирование | 0.4511 | $224,742 | Низкий MAE | Переобучение, низкий R² |

### 2. Лучший результат

**Лучшая модель — явная стратификация по цене**

| Параметр | Значение |
|----------|----------|
| R² | 0.6222 |
| Ценовые сегменты | cheap (< $100k), economy ($100-500k), standard ($500k-2M), luxury (> $2M) |

**Почему выбрана эта модель:**
- Наивысший коэффициент детерминации (R²)
- Стабильность предсказаний на новых данных
- Отсутствие переобучения
- Простота интерпретации

### 3. Что не сработало

| Метод | Причина неудачи |
|-------|-----------------|
| Оптимизация гиперпараметров (Optuna) | Переобучение на валидации |
| Логарифмирование целевой переменной | Ухудшило R² на тесте |
| PCA (снижение размерности) | Потеря информативности |
| Стандартизация данных | Не дала улучшения для Random Forest |

### 4. Рекомендация для продакшена

- Использовать стратифицированную модель из `03_modeling.ipynb`
- Веб-сервис `web_service/app.py` не требует изменений
- Модель готова к работе на реальных данных

### 5. Итоговый вывод

Разработанная модель прогнозирования стоимости жилья на основе явной стратификации по цене достигает коэффициента детерминации R² = 0.6222, что является достаточным для практического использования в агентстве недвижимости. Модель успешно интегрирована в веб-сервис на FastAPI и готова к приёму запросов.